# Tool Calling

User -> LLM -> decides tool call -> Tool Executor -> LLM -> Final Answer

In [1]:
import os
from dotenv import load_dotenv
from groq import Groq

In [5]:
load_dotenv()

True

In [2]:
def get_weather(city: str):
    return f"Weather in {city} is 32°C and sunny."

In [3]:
tools = [
    {
    "type": "function",
     "function": {
        "name": "get_weather",
        "description": "Get the current weather in a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get the weather for."
                }
            },
            "required": ["city"]
        }
     },
    }
]

In [4]:
tool_registry = {
    "get_weather": get_weather
}

In [7]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    tools=tools,
    messages=[
        {"role": "user", "content": "What's the weather in Hyderabad"}
    ]
    )

In [23]:
def run_agent(messages, max_iterations=5):

    for _ in range(max_iterations):

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",  
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        print(f"Model response: {response.choices[0]}")

        message = response.choices[0].message
        print(f"Model response: {message.content}")

        if not message.tool_calls:
            return message.content

        messages.append({
            "role": "assistant",
            "tool_calls": message.tool_calls
        })

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if tool_name not in tool_registry:
                raise Exception(f"Unknown tool: {tool_name}")

            result = tool_registry[tool_name](**arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    raise Exception("Max iterations reached")

In [22]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the weather in Hyderabad?"}
]

response = run_agent(messages)
print(response)

Model response: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='m84nvzf1p', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]))
Model response: None
Tool result: Weather in Hyderabad is 32°C and sunny.
Model response: Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current weather in Hyderabad is 32°C and sunny.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))
Model response: The current weather in Hyderabad is 32°C and sunny.
The current weather in Hyderabad is 32°C and sunny.


In [24]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for real-time information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]

In [26]:
import requests

def get_weather(city: str):
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_resp = requests.get(geo_url, params={"name": city, "count": 1}, timeout=5)
    geo_data = geo_resp.json()
    if "results" not in geo_data:
        return {"error": "City not found"}
    lat = geo_data["results"][0]["latitude"]
    lon = geo_data["results"][0]["longitude"]
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_resp = requests.get(
        weather_url,
        params={
            "latitude": lat,
            "longitude": lon,
            "current_weather": True
        },
        timeout=5
    )
    weather_data = weather_resp.json()

    return weather_data["current_weather"]

In [27]:
def search_web(query: str):
    url = "https://api.duckduckgo.com/"
    response = requests.get(
        url,
        params={
            "q": query,
            "format": "json"
        },
        timeout=5
    )

    data = response.json()

    return {
        "abstract": data.get("Abstract"),
        "related_topics": data.get("RelatedTopics")[:3]
    }

In [28]:
tool_registry = {
    "get_weather": get_weather,
    "search_web": search_web
}

In [29]:
from groq import Groq
import os
import json

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [31]:
def run_agent(messages, max_iterations=5):

    for _ in range(max_iterations):

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        if not message.tool_calls:
            return message.content

        messages.append({
            "role": "assistant",
            "tool_calls": message.tool_calls
        })

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if tool_name not in tool_registry:
                result = {"error": "Unknown tool"}
            else:
                try:
                    result = tool_registry[tool_name](**arguments)
                except Exception as e:
                    result = {"error": str(e)}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    return "Max iterations reached."

In [32]:
messages = [
    {"role": "system", "content": "You are a smart assistant. Use tools when needed."},
    {"role": "user", "content": "What is the current weather in Hyderabad?"}
]

final_response = run_agent(messages)
print(final_response)

The current weather in Hyderabad is mostly cloudy with a temperature of 24.8 degrees Celsius and a wind speed of 6.1 meters per second.


In [33]:
messages = [
    {"role": "system", "content": "You are a smart assistant. Use tools when needed."},
    {"role": "user", "content": "Who is the CEO of Nvidia?"}
]

final_response = run_agent(messages)
print(final_response)

The current CEO of Nvidia is Jensen Huang.
